### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [5]:
### Gemini API Key and Opensource models--Llama3,Gemma2,Mistral

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.8-27b", groq_api_key=groq_api_key)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x0000020D8C48D990>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020D8C5F4C10>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
'''What is happening here is we are setting up how HumanMessage and SystemMessage will be used to interact with the model. The SystemMessage is used to provide instructions to the model, while the HumanMessage is used to provide input from the user. In this case, we are asking the model to translate a sentence from English to French. The result of the translation will be stored in the variable 'result'. 
'''


from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from Emglish to French"),
    HumanMessage(content="Hello, How are you?")
]

result=model.invoke(messages)

In [9]:
result

AIMessage(content='Bonjour, comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 32, 'total_tokens': 39, 'completion_time': 0.013504865, 'completion_tokens_details': None, 'prompt_time': 0.002017837, 'prompt_tokens_details': None, 'queue_time': 0.046540942, 'total_time': 0.015522702}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f710-a1f7-7160-88ed-d9139d9bab58-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 7, 'total_tokens': 39})

In [10]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous ?'

In [11]:
## Using LCEL- chain The Components

chain=model | parser
chain.invoke(messages)


'Bonjour, comment allez-vous ?'

In [12]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [13]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [14]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [15]:
## Chaining together the prompt and the model and the output parser

chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello, How are you?"})


'Bonjour, comment allez-vous ?'